In [1]:
import os

In [2]:
%pwd

'c:\\MLOps\\TextSummarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\MLOps\\TextSummarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_path=PARAMS_FILE_PATH):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [8]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

c:\MLOps\datascienceproject\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### Data Transformation Component

In [18]:
from datasets import load_dataset, load_from_disk


class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)
        target_encodings = self.tokenizer(example_batch['summary'], max_length=128, truncation=True)

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"],
        }

    def convert(self):
        input_dataset_path = os.path.join(self.config.root_dir, "input_dataset")

        if os.path.exists(input_dataset_path):
            dataset_samsum = load_from_disk(input_dataset_path)
        else:
            try:
                dataset_samsum = load_dataset("samsum", split="train")
            except Exception:
                dataset_samsum = load_dataset("knkarthick/dialogsum", split="train")

            os.makedirs(os.path.dirname(input_dataset_path), exist_ok=True)
            dataset_samsum.save_to_disk(input_dataset_path)

        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        os.makedirs(self.config.root_dir, exist_ok=True)
        output_path = os.path.join(self.config.root_dir, "processed_dataset")
        dataset_samsum_pt.save_to_disk(output_path)

In [19]:
%pip install "transformers[sentencepiece]" sentencepiece

config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

Note: you may need to restart the kernel to use updated packages.
[2026-07-29 00:11:10,070: INFO: common: YAML file loaded successfully: config\config.yaml]
[2026-07-29 00:11:10,072: INFO: common: YAML file loaded successfully: params.yaml]
[2026-07-29 00:11:10,073: INFO: common: Directory created at: artifacts]
[2026-07-29 00:11:10,074: INFO: common: Directory created at: artifacts/data_transformation]
[2026-07-29 00:11:11,435: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-29 00:11:11,503: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-07-29 00:11:11,823: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-29 00:11:11,915: INFO

Saving the dataset (1/1 shards): 100%|██████████| 12460/12460 [00:00<00:00, 347817.88 examples/s]
